In [ ]:
# ============================================================
# 03_test_finetuned_model.py
# 목적:
#   파인튜닝된 모델이 5개 항목 형식을 지키는지 테스트
# ============================================================

from unsloth import FastLanguageModel
import torch

FastLanguageModel.for_inference(model)


SYSTEM_PROMPT = """너는 미 육군 교리 기반 RAG 답변 보조자다.
반드시 제공된 검색 근거 범위 안에서만 답변한다.
답변은 다음 5개 항목을 모두 포함해야 한다.

1. 질문 해석
2. 관련 교범 근거
3. 작전적 검토
4. 권고 또는 참모 검토사항
5. 한계사항

답변 본문에는 chunk_id, chunk, 청크 같은 내부 데이터 관리 용어를 절대 사용하지 않는다.
근거가 부족하면 단정하지 말고, 검색된 자료 범위와 추가 확인 필요성을 명시한다."""


def generate_answer(question, context, citation):
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {
            "role": "user",
            "content": f"""[검색 근거]
{context}

[출처 정보]
{citation}

[질문]
{question}

위 검색 근거와 출처 정보만 바탕으로 답변하라.
답변은 반드시 다음 형식을 따른다.

1. 질문 해석
2. 관련 교범 근거
3. 작전적 검토
4. 권고 또는 참모 검토사항
5. 한계사항""",
        },
    ]

    inputs = tokenizer.apply_chat_template(
        messages,
        tokenize=True,
        add_generation_prompt=True,
        return_tensors="pt",
    ).to("cuda")

    outputs = model.generate(
        input_ids=inputs,
        max_new_tokens=900,
        temperature=0.2,
        top_p=0.9,
        repetition_penalty=1.1,
        do_sample=True,
        eos_token_id=tokenizer.eos_token_id,
    )

    decoded = tokenizer.decode(outputs[0], skip_special_tokens=True)
    return decoded


test_question = "작전계획을 수립할 때 지휘관과 참모가 공통적으로 고려해야 할 핵심 사항은 무엇인가?"

test_context = """
작전계획 수립은 지휘관의 의도, 임무분석, 작전환경 이해, 가용 전투력, 위험, 시간, 지휘통제, 지속지원 요소를 통합해 수행해야 한다.
참모는 계획 수립 과정에서 각 전투기능을 통합하고, 지휘관이 결심할 수 있도록 관련 판단자료를 제공해야 한다.
"""

test_citation = "FM 5-0 Planning and Orders Production; FM 6-0 Commander and Staff Organization and Operations"

answer = generate_answer(test_question, test_context, test_citation)
print(answer)